# Supervisor Walkthrough

This is the shortest path through the alpha evidence package.


## This Notebook Should Answer
- What problem is being solved?
- How does the method work?
- What was actually run?
- What do the metrics and figures mean?
- What are the main limitations?
- Is alpha approval justified?


## Method Summary
The system compares a probe recording against claimed-speaker enrollment recordings, estimates spoof likelihood, fuses those signals into a three-way decision, and produces segment-level explanation cues so the decision can be inspected rather than simply accepted.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RUN_OVERRIDE = None  # Set to an absolute run path to force a specific run.

def dataset_mode(run_path: Path) -> str:
    review_path = run_path / 'reports' / 'dataset_review.json'
    if not review_path.exists():
        return 'unknown'
    return json.loads(review_path.read_text()).get('dataset_mode', 'unknown')

RUNS = sorted((ROOT / 'outputs' / 'runs').glob('*fusion_plus_interpretable_features'))
REAL_RUNS = [run for run in RUNS if dataset_mode(run) not in {'demo', 'unknown'}]
SELECTED_RUN = Path(RUN_OVERRIDE) if RUN_OVERRIDE else (REAL_RUNS[-1] if REAL_RUNS else (RUNS[-1] if RUNS else None))
print('Selected fusion run:', SELECTED_RUN)
print('Selected dataset mode:', dataset_mode(SELECTED_RUN) if SELECTED_RUN else 'none')


In [ ]:
report = (SELECTED_RUN / 'reports' / 'supervisor_report.md').read_text() if SELECTED_RUN else 'Run scripts/generate_supervisor_report.py first.'
plot_inventory = pd.read_csv(SELECTED_RUN / 'tables' / 'plot_inventory.csv') if SELECTED_RUN else pd.DataFrame()
alpha_checklist = json.loads((SELECTED_RUN / 'reports' / 'alpha_exit_checklist.json').read_text()) if SELECTED_RUN else {}
dataset_review = json.loads((SELECTED_RUN / 'reports' / 'dataset_review.json').read_text()) if SELECTED_RUN and (SELECTED_RUN / 'reports' / 'dataset_review.json').exists() else {}
display(Markdown(report))
display(Markdown('## Dataset Review Snapshot'))
pd.DataFrame([dataset_review]) if dataset_review else pd.DataFrame(), plot_inventory.head(), alpha_checklist


In [ ]:
if SELECTED_RUN:
    for name in [
        'confusion_matrix.png',
        'normalized_confusion_matrix.png',
        'reliability_diagram.png',
        'threshold_sweep_heatmap.png',
        'ablation_summary.png',
        'waveform_with_suspicious_segments.png',
    ]:
        path = SELECTED_RUN / 'plots' / name
        if path.exists():
            display(Image(filename=str(path)))


## Supervisor Verdict Template
Alpha approval is justified when the run artifacts are complete, the notebooks and reports load cleanly from saved files, the figures are interpretable, and the claims remain modest and explicitly limited to methodology and infrastructure readiness.
